# 🌱 Consultas de análisis sobre Azure SQL Database

Ejecuta las 10 consultas de `sql/02_consultas_analisis.sql` (versión T-SQL) contra la base de datos
que carga la Azure Function cada mes.

**Antes de empezar:**
1. Copia `.env.example` como `.env` y rellena `SQL_CADENA_CONEXION` (el archivo `.env` no se sube a GitHub).
2. Tu IP debe estar en el firewall del servidor SQL (`scripts/desplegar.ps1 -IpCliente <tu IP>`).
3. La primera conexión puede tardar hasta 1 minuto: la base de datos gratuita se pausa sola cuando no se usa.

In [ ]:
import re
import sys
from pathlib import Path

import pandas as pd

# Rutas (funciona tanto si se abre desde la raíz como desde la carpeta notebooks/)
RAIZ = Path.cwd() if (Path.cwd() / "sql").exists() else Path.cwd().parent
sys.path.insert(0, str(RAIZ))   # para poder importar src/

import os
from src.carga_sql import conectar
from src.pipeline import leer_archivo_env

leer_archivo_env(RAIZ / ".env")
conexion = conectar(os.environ["SQL_CADENA_CONEXION"])
print("Conectado a Azure SQL")

In [ ]:
# ---------- Consultas SQL (mismo sistema que el notebook original) ----------
def leer_consultas(ruta: Path) -> dict[str, str]:
    """Separa el archivo .sql en consultas usando las marcas '-- @nombre: xxx'."""
    partes = re.split(r"^-- @nombre:\s*(\w+)\s*$", ruta.read_text(encoding="utf-8"), flags=re.MULTILINE)
    return {nombre: sql.strip() for nombre, sql in zip(partes[1::2], partes[2::2])}

CONSULTAS = leer_consultas(RAIZ / "sql" / "02_consultas_analisis.sql")


def consulta(nombre: str) -> pd.DataFrame:
    """Ejecuta una consulta por su nombre y devuelve el resultado como DataFrame."""
    cursor = conexion.cursor()
    cursor.execute(CONSULTAS[nombre])
    columnas = [c[0] for c in cursor.description]
    return pd.DataFrame([tuple(fila) for fila in cursor.fetchall()], columns=columnas)


print(f"{len(CONSULTAS)} consultas disponibles:", list(CONSULTAS))

## Comprobación de la carga

In [ ]:
cursor = conexion.cursor()
for tabla in ["dim_pais", "dim_cultivo", "fact_superficie", "fact_pct_sau", "fact_operadores", "fact_ganaderia"]:
    cursor.execute(f"SELECT COUNT(*) FROM {tabla}")
    print(f"{tabla:<16} {cursor.fetchone()[0]:>7,} filas")

## 1. Evolución de la superficie ecológica en España

In [ ]:
consulta("q01_evolucion_espana")

## 2. Ranking de la UE por hectáreas

In [ ]:
consulta("q03_ranking_ue_superficie").head(10)

## 3. Todas las consultas

In [ ]:
for nombre in CONSULTAS:
    resultado = consulta(nombre)
    print(f"
=== {nombre}: {len(resultado)} filas ===")
    display(resultado.head())

In [ ]:
conexion.close()